# SaansCare – Model 3: Vehicle Emission Classifier + Green Score Pipeline
**Goal:** from the details a vehicle owner knows (type, fuel, age, engine size, odometer, service history) predict
(A) the **emission tier** (Low / Medium / High) and (B) a **Green Score** from 0 to 100 for the downloadable certificate.
**Algorithms compared:** Logistic Regression, Decision Tree, Random Forest, **XGBoost** (classifier); Ridge, Random Forest, **XGBoost** (Green Score regressor).
**Metrics:** accuracy, precision, recall, F1 (per class), confusion matrix; RMSE, MAE, R² for the score.

**How to run:** run the cells top to bottom. Step 2 asks you for `vehicle_emission.csv` / `.xlsx`.

## 1. Setup
*What we are doing:* importing libraries and creating the output folder.

In [ ]:
import sys, subprocess, os, json, warnings, zipfile, time
from pathlib import Path
warnings.filterwarnings("ignore")

def _ensure(pkg, import_name=None):
    """Install a package only if it is missing (Google Colab already has most of them)."""
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_ensure("xgboost"); _ensure("openpyxl"); _ensure("joblib")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
                             classification_report, mean_squared_error, mean_absolute_error, r2_score)
from xgboost import XGBClassifier, XGBRegressor

RANDOM_STATE = 42                      # fixed seed -> reproducible results
np.random.seed(RANDOM_STATE)
OUT_DIR = Path("saanscare_outputs/model3_vehicle_emission")   # everything this notebook produces is saved here
OUT_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "grid.alpha": 0.3})
print("Setup complete. Outputs will be saved in:", OUT_DIR.resolve())

In [ ]:
def plot_confusion(y_true, y_pred, labels, title, path=None):
    """Confusion matrix as counts and as row-percentages (recall per class)."""
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(labels))))
    cmn = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(5.8, 5))
    ax.imshow(cmn, cmap="Greens", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{cm[i, j]:,}\n{cmn[i, j]*100:.0f}%", ha="center", va="center",
                    color="white" if cmn[i, j] > 0.55 else "black", fontsize=9)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(title); ax.grid(False)
    plt.tight_layout()
    if path: plt.savefig(path, dpi=130)
    plt.show()

def warning_rates(y_true, y_pred):
    """For ordered risk classes: under-warning (predicted LOWER risk than reality) is the dangerous error."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return {"exact": float((y_pred == y_true).mean()),
            "under_warning": float((y_pred < y_true).mean()),
            "over_warning": float((y_pred > y_true).mean())}

def clf_scores(y_true, y_pred):
    return {"accuracy": accuracy_score(y_true, y_pred),
            "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0)}

def build_pre(num_cols, cat_cols, scale=True):
    """Preprocessing: impute missing values, scale numbers, one-hot encode text categories."""
    num_steps = [("imp", SimpleImputer(strategy="median"))]
    if scale: num_steps.append(("sc", StandardScaler()))
    cat_steps = [("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]
    return ColumnTransformer([("num", Pipeline(num_steps), num_cols), ("cat", Pipeline(cat_steps), cat_cols)])

def to_jsonable(o):
    if isinstance(o, dict): return {str(k): to_jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [to_jsonable(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    return o

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(to_jsonable(obj), f, indent=2)

def package_outputs(out_dir, zip_name):
    """Zip the output folder; in Google Colab this also starts the download."""
    zpath = Path(zip_name)
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for p in Path(out_dir).rglob("*"):
            if p.is_file(): z.write(p, p.relative_to(Path(out_dir).parent))
    print("Packaged:", zpath.resolve(), f"({zpath.stat().st_size/1e6:.1f} MB)")
    try:
        from google.colab import files
        files.download(str(zpath))
    except ImportError:
        pass

## 2. Load the dataset (the notebook asks you for it)
*What we are doing:* asking for the dataset file and checking that the expected columns exist.

In [ ]:
def load_dataset(label, required_cols):
    """Ask the user for the dataset, read it, and check that the expected columns are present."""
    path = os.environ.get("SAANSCARE_DATASET")            # optional shortcut: skips the prompt
    if not path:
        try:
            from google.colab import files                 # running in Google Colab -> upload dialog
            print(f"Upload your {label} file (.csv or .xlsx) ...")
            uploaded = files.upload()
            if not uploaded:
                raise FileNotFoundError("No file was uploaded.")
            path = next(iter(uploaded))
        except ImportError:                                # running in VS Code / Jupyter -> ask for the path
            path = input(f"Enter the full path of your {label} file (.csv or .xlsx): ").strip().strip('"').strip("'")
    df = pd.read_excel(path) if path.lower().endswith((".xlsx", ".xls")) else pd.read_csv(path)
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"The file is missing these required columns: {missing}\nColumns found: {list(df.columns)}")
    print(f"Loaded {Path(path).name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
    return df

REQUIRED = ["vehicle_type", "make", "model", "year", "fuel_type", "engine_cc", "weight_kg", "odometer_km", "months_since_service",
            "months_since_smoke_test", "emission_tier", "green_score"]
df = load_dataset("vehicle emission dataset", REQUIRED)
df.head()

## 3. Data understanding (EDA)
*What we are doing:* checking class balance, vehicle mix, missing values and how emissions relate to age and fuel.

In [ ]:
TIERS = ["Low", "Medium", "High"]
print("Missing values:\n", df.isna().sum()[lambda s: s > 0].to_string() or "none")
print("\nTier distribution (%):"); print((df["emission_tier"].value_counts(normalize=True).reindex(TIERS) * 100).round(1).to_string())
display(df.describe().T.round(2))
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
df["emission_tier"].value_counts().reindex(TIERS).plot(kind="bar", ax=ax[0], color=["#86EFAC", "#FDE047", "#DC2626"], title="Tier balance")
pd.crosstab(df["vehicle_type"], df["emission_tier"], normalize="index")[TIERS].plot(kind="bar", stacked=True, ax=ax[1], color=["#86EFAC", "#FDE047", "#DC2626"], title="Tier by vehicle type")
ax[2].hist(df["green_score"], bins=30, color="#16A34A"); ax[2].set_title("Green Score distribution")
plt.tight_layout(); plt.show()

## 4. Data cleaning
*What we are doing:* removing duplicates and impossible rows (negative values, future model years), capping extreme measurements, and dropping columns that would leak the answer.

In [ ]:
n0 = len(df)
if "vehicle_id" in df.columns:
    df = df.drop_duplicates("vehicle_id")
df = df.drop_duplicates().reset_index(drop=True)
print("Duplicates removed:", n0 - len(df))

REF_YEAR = pd.Timestamp.today().year
ok = (df["year"].between(1980, REF_YEAR)) & (df["engine_cc"] > 0) & (df["odometer_km"] >= 0) & (df["weight_kg"] > 0) & df["emission_tier"].isin(TIERS)
print("Invalid rows dropped:", int((~ok).sum()))
df = df[ok].reset_index(drop=True)

for col in ["vehicle_type", "make", "model", "fuel_type", "emission_standard"]:
    if col in df: df[col] = df[col].astype(str).str.strip()

MEASURED = [c for c in ["co_g_km", "hc_g_km", "nox_g_km", "pm_g_km", "smoke_opacity_pct", "co2_g_km", "fuel_l_100km"] if c in df.columns]
for col in MEASURED:        # winsorize extreme sensor readings at the 99.9th percentile
    df[col] = df[col].clip(upper=df[col].quantile(0.999))
print("Measured emission columns found:", MEASURED)
print("Clean shape:", df.shape)

## 5. Feature engineering and feature sets
*What we are doing:* turning the model year into vehicle **age**, and defining two input sets:
* **Profile features (deployed):** what an owner can type into the app (type, make, model, fuel, age, engine, weight, odometer, service and smoke-test history).
* **Full features (reference):** profile + measured emissions from a test. Shows how much a real measurement adds.

In [ ]:
df["age"] = REF_YEAR - df["year"]
df["km_per_year"] = df["odometer_km"] / df["age"].clip(lower=1)
CAT = [c for c in ["vehicle_type", "make", "model", "fuel_type", "emission_standard"] if c in df.columns]
PROFILE_NUM = ["age", "engine_cc", "weight_kg", "odometer_km", "km_per_year", "months_since_service", "months_since_smoke_test"]
FULL_NUM = PROFILE_NUM + MEASURED
FEATURE_SETS = {"profile": (PROFILE_NUM, CAT), "full (with measured emissions)": (FULL_NUM, CAT)}
LABEL_TO_ID = {l: i for i, l in enumerate(TIERS)}
y_cls = df["emission_tier"].map(LABEL_TO_ID)
y_reg = df["green_score"]
print("Profile inputs:", PROFILE_NUM + CAT)

## 6. Train / validation / test split (stratified)
*What we are doing:* 70% / 15% / 15%, stratified on the emission tier so each part has the same class mix. The same row split is used for the Green Score model.

In [ ]:
idx = np.arange(len(df))
i_tr, i_tmp = train_test_split(idx, test_size=0.30, stratify=y_cls, random_state=RANDOM_STATE)
i_va, i_te = train_test_split(i_tmp, test_size=0.50, stratify=y_cls.iloc[i_tmp], random_state=RANDOM_STATE)
print(f"train {len(i_tr):,} | validation {len(i_va):,} | test {len(i_te):,}")

## 7. Preprocessing pipeline
*What we are doing:* a transformer that fills missing numbers (e.g. smoke opacity exists only for diesel), scales numbers and one-hot encodes categories.
Built per feature set inside the model pipeline.

In [ ]:
def make_cls_models(num, cat):
    pre_s, pre_t = build_pre(num, cat, True), build_pre(num, cat, False)
    return {
        "Logistic Regression": Pipeline([("pre", pre_s), ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))]),
        "Decision Tree": Pipeline([("pre", pre_t), ("model", DecisionTreeClassifier(max_depth=12, class_weight="balanced", random_state=RANDOM_STATE))]),
        "Random Forest": Pipeline([("pre", pre_t), ("model", RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE))]),
        "XGBoost": Pipeline([("pre", pre_t), ("model", XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.9, colsample_bytree=0.9,
                                                                      objective="multi:softprob", eval_metric="mlogloss", tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE))]),
    }
print("Pipelines defined")

## 8. Task A – emission tier classifier: training and comparison
*What we are doing:* training the four classifiers on both feature sets and comparing them on the validation set.

In [ ]:
rows, fitted = [], {}
for fs_name, (num, cat) in FEATURE_SETS.items():
    X = df[num + cat]
    Xtr, Xva = X.iloc[i_tr], X.iloc[i_va]; ytr, yva = y_cls.iloc[i_tr], y_cls.iloc[i_va]
    w = compute_sample_weight("balanced", ytr)
    for name, pipe in make_cls_models(num, cat).items():
        t0 = time.time()
        if name == "XGBoost": pipe.fit(Xtr, ytr, model__sample_weight=w)
        else: pipe.fit(Xtr, ytr)
        pv = pipe.predict(Xva); s = clf_scores(yva, pv); s.update(warning_rates(yva, pv))
        s.update({"feature_set": fs_name, "model": name, "seconds": round(time.time() - t0, 1)})
        rows.append(s); fitted[(fs_name, name)] = pipe
comparison = pd.DataFrame(rows).set_index(["feature_set", "model"]).round(3).sort_values("f1_macro", ascending=False)
comparison

## 9. Hyperparameter tuning (deployed profile model)
*What we are doing:* tuning XGBoost and Random Forest on the **profile** features (the ones the app can actually collect) with 3-fold cross-validation on macro-F1.

In [ ]:
num, cat = FEATURE_SETS["profile"]
Xp = df[num + cat]; Xtr, Xva, Xte = Xp.iloc[i_tr], Xp.iloc[i_va], Xp.iloc[i_te]
ytr, yva, yte = y_cls.iloc[i_tr], y_cls.iloc[i_va], y_cls.iloc[i_te]
w_tr = compute_sample_weight("balanced", ytr)
base = make_cls_models(num, cat)
def tune(pipe, space, n_iter=10, fit_params=None):
    s = RandomizedSearchCV(pipe, space, n_iter=n_iter, cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE), scoring="f1_macro", random_state=RANDOM_STATE, n_jobs=1)
    return s.fit(Xtr, ytr, **(fit_params or {}))
t0 = time.time()
s_xgb = tune(base["XGBoost"], {"model__n_estimators": [200, 300, 500], "model__max_depth": [4, 6, 8], "model__learning_rate": [0.05, 0.1, 0.2],
                               "model__subsample": [0.8, 0.9, 1.0], "model__colsample_bytree": [0.7, 0.9, 1.0], "model__min_child_weight": [1, 3, 5]},
             fit_params={"model__sample_weight": w_tr})
s_rf = tune(base["Random Forest"], {"model__n_estimators": [200, 400], "model__max_depth": [None, 15, 25], "model__min_samples_leaf": [1, 2, 4], "model__max_features": ["sqrt", 0.5]}, n_iter=6)
print(f"Tuning took {time.time()-t0:.0f}s")
cands = {"XGBoost (tuned)": s_xgb.best_estimator_, "Random Forest (tuned)": s_rf.best_estimator_}
for n_, m_ in cands.items(): print(f"{n_:22s} validation f1_macro = {f1_score(yva, m_.predict(Xva), average='macro'):.3f}")
best_name = max(cands, key=lambda n_: f1_score(yva, cands[n_].predict(Xva), average="macro")); best_cls = cands[best_name]
print("Selected:", best_name)

## 10. Task A – final evaluation on the test set
*What we are doing:* evaluating the selected profile model once on unseen vehicles: accuracy, precision, recall, F1 per class, confusion matrix, and results per vehicle type.

In [ ]:
pred = best_cls.predict(Xte)
print(classification_report(yte, pred, target_names=TIERS, digits=3))
cls_scores = clf_scores(yte, pred); cls_scores.update(warning_rates(yte, pred)); print({k: round(v, 4) for k, v in cls_scores.items()})
plot_confusion(yte, pred, TIERS, f"Emission tier – {best_name} (test)", OUT_DIR / "confusion_matrix_test.png")
by_type = pd.DataFrame({"type": df.iloc[i_te]["vehicle_type"].values, "ok": (pred == yte.values)}).groupby("type")["ok"].agg(["mean", "count"]).round(3)
print("\nAccuracy by vehicle type:"); display(by_type)
full_pipe = fitted[("full (with measured emissions)", "XGBoost")]
fnum, fcat = FEATURE_SETS["full (with measured emissions)"]
p_full = full_pipe.predict(df[fnum + fcat].iloc[i_te])
print(f"Reference: with measured emissions macro-F1 = {f1_score(yte, p_full, average='macro'):.3f} vs profile-only = {cls_scores['f1_macro']:.3f}")

## 11. Task B – Green Score regression (0-100)
*What we are doing:* predicting the Green Score from the same profile features. Metrics: **RMSE** and **MAE** (average error in score points) and **R²**.

In [ ]:
ytr_r, yva_r, yte_r = y_reg.iloc[i_tr], y_reg.iloc[i_va], y_reg.iloc[i_te]
pre_s, pre_t = build_pre(num, cat, True), build_pre(num, cat, False)
reg_models = {
    "Ridge": Pipeline([("pre", pre_s), ("model", Ridge(alpha=1.0))]),
    "Random Forest": Pipeline([("pre", pre_t), ("model", RandomForestRegressor(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE))]),
    "XGBoost": Pipeline([("pre", pre_t), ("model", XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE))]),
}
def reg_score(y, p):
    p = np.clip(p, 0, 100)
    return {"RMSE": float(np.sqrt(mean_squared_error(y, p))), "MAE": float(mean_absolute_error(y, p)), "R2": float(r2_score(y, p))}
reg_rows = {}
for n_, m_ in reg_models.items():
    m_.fit(Xtr, ytr_r); reg_rows[n_] = reg_score(yva_r, m_.predict(Xva))
display(pd.DataFrame(reg_rows).T.round(3).sort_values("RMSE"))
best_reg_name = min(reg_rows, key=lambda k: reg_rows[k]["RMSE"]); best_reg = reg_models[best_reg_name]
best_reg = clone(best_reg).fit(pd.concat([Xtr, Xva]), pd.concat([ytr_r, yva_r]))
reg_test = reg_score(yte_r, best_reg.predict(Xte))
print("Selected:", best_reg_name, "| test:", {k: round(v, 3) for k, v in reg_test.items()})
pr = np.clip(best_reg.predict(Xte), 0, 100)
plt.scatter(yte_r, pr, s=5, alpha=.4); plt.plot([0, 100], [0, 100], "r--"); plt.xlabel("actual Green Score"); plt.ylabel("predicted")
plt.title(f"Green Score – {best_reg_name} (test)"); plt.savefig(OUT_DIR / "green_score_test.png", dpi=130); plt.show()

## 12. Feature importance
*What we are doing:* checking what drives the emission tier (expect: emission standard, age, fuel type, odometer, service history).

In [ ]:
pi = permutation_importance(best_cls, Xte, yte, scoring="f1_macro", n_repeats=5, random_state=RANDOM_STATE, n_jobs=1)
pd.Series(pi.importances_mean, index=Xte.columns).sort_values().plot(kind="barh", color="#16A34A", title="Permutation importance – emission tier (drop in macro-F1)")
plt.tight_layout(); plt.savefig(OUT_DIR / "feature_importance.png", dpi=130); plt.show()

## 13. Save the models
*What we are doing:* saving both pipelines (preprocessing + model) with metadata for the FastAPI service.

In [ ]:
meta = {"model": f"Vehicle emission tier ({best_name}) + Green Score ({best_reg_name})", "tier_labels": TIERS, "profile_features": PROFILE_NUM + CAT,
        "num_features": PROFILE_NUM, "cat_features": CAT, "reference_year": REF_YEAR, "metrics_tier_test": cls_scores, "metrics_green_score_test": reg_test,
        "validation_comparison": comparison.reset_index().to_dict("records"),
        "note": "Tier and Green Score derive from a documented rule on emissions; trained on simulated/mixed data - not a regulatory test.",
        "version": "1.0", "random_state": RANDOM_STATE}
joblib.dump({"tier_pipeline": best_cls, "green_score_pipeline": best_reg, "meta": meta}, OUT_DIR / "vehicle_emission_models.joblib")
save_json(meta, OUT_DIR / "vehicle_emission_metadata.json")
comparison.to_csv(OUT_DIR / "model_comparison_validation.csv")
print("Saved:", sorted(p.name for p in OUT_DIR.iterdir()))

## 14. Inference demo (how the web app will use it)
*What we are doing:* loading the saved file and scoring one registered vehicle – this produces the data shown on the Green Score certificate.

In [ ]:
def score_vehicle(vehicle: dict):
    b = joblib.load(OUT_DIR / "vehicle_emission_models.joblib"); m = b["meta"]
    v = dict(vehicle); v["age"] = m["reference_year"] - v["year"]; v["km_per_year"] = v["odometer_km"] / max(v["age"], 1)
    row = pd.DataFrame([v])[m["profile_features"]]
    p = b["tier_pipeline"].predict_proba(row)[0]
    return {"emission_tier": m["tier_labels"][int(p.argmax())], "tier_probabilities": {l: round(float(x), 3) for l, x in zip(m["tier_labels"], p)},
            "green_score": round(float(np.clip(b["green_score_pipeline"].predict(row)[0], 0, 100)), 1)}

example = {"vehicle_type": "car", "make": "Toyota", "model": "Corolla", "fuel_type": "petrol", "emission_standard": "Euro-2", "year": 2012,
           "engine_cc": 1600, "weight_kg": 1250, "odometer_km": 165000, "months_since_service": 14, "months_since_smoke_test": 20}
print(score_vehicle({k: v for k, v in example.items() if k in PROFILE_NUM + CAT or k == "year"}))

## 14b. Package and download
*What we are doing:* zipping the models, metadata, metrics and plots.

In [ ]:
package_outputs(OUT_DIR, "model3_vehicle_emission_outputs.zip")

### Notes for the viva
* Both the tier and the Green Score come from the same documented rule on emissions, so they are strongly related by construction. `green_score` is excluded from the tier inputs to avoid leakage.
* The deployed model uses only what an owner can enter; the "full" feature set shows how much an actual emission-test measurement would add.
* Train on the real **US EPA / UK DVSA** vehicle data as well (same column names) before claiming real-world accuracy.